# BI2K Bronchoscopic Image Classification

Trains a 3-class classifier (**normal / benign / malignant**) on the [BI2K Bronchoscopic Dataset](https://www.kaggle.com/datasets/timesxy/bronchoscopic-dataset-bi2k-3-classification) using transfer learning with a ResNet backbone.

**Dataset:** 2,900 bronchoscopic images — 1,350 normal, 600 benign lesion, 950 malignant lesion (per the dataset's Kaggle page / the PKDN paper it's drawn from).

**Assumed folder layout after download** (standard `ImageFolder` layout — adjust `DATA_DIR` / class names below if the actual download differs):
```
data/
├── normal/
├── benign/
└── malignant/
```

**Pipeline:** load & split data → augment → fine-tune a pretrained ResNet18 → validate each epoch, keep best checkpoint → evaluate on a held-out test set (classification report, confusion matrix) → Grad-CAM sanity check on a few predictions.


## 1. Setup

If you're on the RTX 5070 Ti (Blackwell) box, make sure you have a **cu128** PyTorch build — the default `pip install torch` may give you an older CUDA build that won't run on Blackwell:

```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
pip install kagglehub scikit-learn matplotlib seaborn grad-cam
```


In [ ]:
import os, random, copy, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Get the dataset

If you already have BI2K on disk (e.g. `C:\Users\Omen Max\Datasets\Bronchoscopic Datasets\BI2K_3`), skip straight to **Option B** below and set `DATA_DIR` to that path. Otherwise, **Option A** downloads it via `kagglehub` (requires a Kaggle account + API token, or Kaggle login in-browser the first time).


In [ ]:
# Option A: download with kagglehub (skip this cell if you already have the data locally)
# import kagglehub
# path = kagglehub.dataset_download("timesxy/bronchoscopic-dataset-bi2k-3-classification")
# print("Dataset downloaded to:", path)
# DATA_DIR = Path(path)


In [ ]:
# Option B: point directly at your local copy
DATA_DIR = Path(r"C:\Users\Omen Max\Datasets\Bronchoscopic Datasets\BI2K_3")

# Inspect the folder structure so we can confirm class names / nesting before building the dataset
def show_tree(root, max_depth=3):
    root = Path(root)
    if not root.exists():
        print("Path does not exist:", root); return
    for dirpath, dirnames, filenames in os.walk(root):
        rel_depth = Path(dirpath).relative_to(root).parts
        if len(rel_depth) > max_depth:
            continue
        indent = "  " * len(rel_depth)
        print(f"{indent}{Path(dirpath).name}/  ({len(filenames)} files)")

show_tree(DATA_DIR)


In [ ]:
# Your folders are named 0/1/2, each containing nested subfolders of images
# (rather than images sitting directly inside 0/, 1/, 2/). torchvision's ImageFolder
# walks each top-level class folder recursively, so this nesting is handled automatically —
# you don't need to flatten anything. We just need to confirm which numeric folder is which class.

IMAGE_ROOT = DATA_DIR

class_dirs = sorted([d for d in Path(IMAGE_ROOT).iterdir() if d.is_dir()], key=lambda d: d.name)
print("Detected class folders:", [d.name for d in class_dirs])
for d in class_dirs:
    n = sum(1 for _ in d.rglob("*") if _.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
    print(f"  {d.name}: {n} images (recursive)")


## 3. Transforms & datasets

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # mild — keep diagnostic features intact
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ImageFolder recursively walks each top-level folder (0/, 1/, 2/) for images regardless of
# how deep they're nested inside, so no flattening is needed. It just labels classes by the
# numeric folder name, so map those to readable labels once you confirm which is which from
# the counts printed above (BI2K is ~1350 normal / 600 benign / 950 malignant).
full_dataset = datasets.ImageFolder(root=str(IMAGE_ROOT), transform=eval_tf)

# EDIT THIS once you've matched folder counts to the paper's class sizes:
CLASS_LABEL_MAP = {"0": "normal", "1": "benign", "2": "malignant"}  # <-- verify against counts above
full_dataset.classes = [CLASS_LABEL_MAP.get(c, c) for c in full_dataset.classes]
class_names = full_dataset.classes

print("Classes:", class_names, "-> indices", full_dataset.class_to_idx)
print("Total images:", len(full_dataset))


In [ ]:
# Stratified-ish split via random_split (70/15/15). For strict stratification swap in
# sklearn.model_selection.train_test_split on full_dataset.samples instead.
n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset, test_subset = random_split(full_dataset, [n_train, n_val, n_test], generator=generator)

# Apply the augmenting transform only to the training subset
train_subset.dataset = copy.copy(full_dataset)
train_subset.dataset.transform = train_tf

print(f"Train: {len(train_subset)}  Val: {len(val_subset)}  Test: {len(test_subset)}")

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_subset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
# Class imbalance check (BI2K is roughly 1350/600/950 normal/benign/malignant)
labels = [full_dataset.samples[i][1] for i in range(len(full_dataset.samples))]
counts = np.bincount(labels, minlength=len(class_names))
for name, c in zip(class_names, counts):
    print(f"{name}: {c}")

plt.figure(figsize=(5,4))
sns.barplot(x=class_names, y=counts)
plt.title("Class distribution (full dataset)")
plt.ylabel("count")
plt.show()


## 4. Model

ResNet18 pretrained on ImageNet, fine-tuned end-to-end. A class-weighted loss compensates for the imbalance (benign lesion is the minority class).

In [ ]:
def build_model(num_classes, freeze_backbone=False):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    in_feats = model.fc.in_features
    model.fc = nn.Linear(in_feats, num_classes)
    return model

model = build_model(num_classes=len(class_names)).to(DEVICE)

# Inverse-frequency class weights to counter imbalance
class_weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32).to(DEVICE)
print("Class weights:", dict(zip(class_names, class_weights.tolist())))

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)


## 5. Training loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            if is_train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)

    return total_loss / total, correct / total


In [ ]:
EPOCHS = 25
PATIENCE = 7  # early stopping
CKPT_PATH = "bi2k_best_model.pt"

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss); history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss);     history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train_loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val_loss {val_loss:.4f} acc {val_acc:.4f} | "
          f"{time.time()-t0:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({"model_state": model.state_dict(), "class_names": class_names}, CKPT_PATH)
        print(f"  -> saved new best model (val_loss {val_loss:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).")
            break


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()


## 6. Evaluate on the held-out test set

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        out = model(x)
        probs = torch.softmax(out, dim=1)
        preds = probs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix — test set")
plt.tight_layout(); plt.show()


## 7. Grad-CAM sanity check

Quick visual check that the model is attending to lesion regions rather than image borders/artifacts — useful before trusting the numbers above. Requires `pip install grad-cam`.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

target_layer = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layer)

def denormalize(img_tensor):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

n_show = 6
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3.5))
shown = 0
for x, y in test_loader:
    for i in range(x.size(0)):
        if shown >= n_show:
            break
        img_t = x[i:i+1].to(DEVICE)
        grayscale_cam = cam(input_tensor=img_t)[0]
        rgb_img = denormalize(x[i])
        vis = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

        pred = model(img_t).argmax(1).item()
        axes[shown].imshow(vis)
        axes[shown].set_title(f"true: {class_names[y[i]]}\npred: {class_names[pred]}", fontsize=9)
        axes[shown].axis("off")
        shown += 1
    if shown >= n_show:
        break
plt.tight_layout(); plt.show()


## Notes / next steps

- The split above is a random 70/15/15 hold-out, not stratified by class — swap in `sklearn.model_selection.train_test_split(..., stratify=labels)` if you want exact per-class ratios preserved in each split.
- If val accuracy plateaus early, try unfreezing fewer layers first (`freeze_backbone=True`, then fine-tune the head only for a few epochs before unfreezing everything), or swap `resnet18` for `resnet50` / `efficientnet_b0` from `torchvision.models`.
- Given the class imbalance (benign is the minority class), also watch **per-class recall** in the classification report above, not just overall accuracy.
- `bi2k_best_model.pt` holds both the weights and `class_names`, so it's self-contained for inference later.
